# Data Engine

In [2]:
import yfinance as yf
import pandas as pd

class yahoo_data:

    @staticmethod
    def OCHLV(ticker, start_date, end_date):
        data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True)
        # Si viene con MultiIndex, lo aplanamos
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.droplevel(1)
        return data
    
    @staticmethod
    def closed_prices(tickets, start_date, end_date):
        #Si las fechas son válidas entonces
        if tickets and start_date < end_date:
            
            #Descarga los precios de cierre ajustados de los tickets en las fechas seleccionadas
            raw = yf.download(tickets, start=start_date, end=end_date, auto_adjust=True)["Close"]
            #Los precios ajsutados son ideales cuando hacemos análisis cuantitativo, pero para contable no
            
            valid_tickets = [] #Lista donde ponemos los simbolos
    
            #Por cada ticket que tengamos en tickets si no esta ponemos que no fue encontrado
            for ticket in tickets: 
                if ticket not in raw.columns or raw[ticket].dropna().empty:
                    print(f"El ticket '{ticket}' no fue encontrado o no tiene datos.")
                else: #si se encuentra lo agregamos a la lista
                    valid_tickets.append(ticket)
    
            # Filtrar activos válidos
            data = raw[valid_tickets].dropna(axis=1, how='all')
            
            return data
    
    @staticmethod
    def get_betas(tickers):
        betas = {}
        for ticker in tickers:
            asset = yf.Ticker(ticker)
            beta_value = asset.info.get("beta")
            betas[ticker] = beta_value
    
        df_betas = pd.DataFrame.from_dict(betas, orient='index', columns=['Beta'])
        
        return df_betas

In [3]:
yahoo_data.OCHLV('AAPL','2023-12-17', '2025-12-17') 

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2023-12-18,194.057327,194.790410,192.571361,194.255453,55751900
2023-12-19,195.097519,195.107420,194.057339,194.324817,40714100
2023-12-20,193.007248,195.830575,193.007248,195.057874,52242800
2023-12-21,192.858658,195.236214,191.689705,194.265386,46482500
2023-12-22,191.788742,193.581806,191.164631,193.353947,37149600
...,...,...,...,...,...
2025-12-10,278.779999,279.750000,276.440002,277.750000,33038300
2025-12-11,278.029999,279.589996,273.809998,279.100006,33248000
2025-12-12,278.279999,279.220001,276.820007,277.899994,39532900


In [4]:
yahoo_data.closed_prices(['AAPL', 'MSFT', 'AMZN'],'2023-12-17', '2025-12-17')

[*********************100%***********************]  3 of 3 completed


Ticker,AAPL,MSFT,AMZN
Date,,,
2023-12-18,194.057327,367.175446,154.070007
2023-12-19,195.097519,367.776459,153.789993
2023-12-20,193.007248,365.175262,152.119995
2023-12-21,192.858658,368.052399,153.839996
2023-12-22,191.788742,369.077087,153.419998
...,...,...,...
2025-12-10,278.779999,478.559998,231.779999
2025-12-11,278.029999,483.470001,230.279999
2025-12-12,278.279999,478.529999,226.190002


# Returns Engine

In [5]:
import numpy as np

class returns:

    @staticmethod
    def simple(closing_prices):
        return closing_prices.pct_change().dropna()
        
    @staticmethod
    def logaritmic(closing_prices):
        return np.log(closing_prices / closing_prices.shift(1)).dropna()
        #Logaritmo Natural(Precio actual / precio anterior)

In [6]:
returns.logaritmic(
    yahoo_data.closed_prices(
    ['AAPL', 'MSFT', 'AMZN'],
    '2023-12-17', 
    '2025-12-17')
           )

[*********************100%***********************]  3 of 3 completed


Ticker,AAPL,MSFT,AMZN
Date,,,
2023-12-19,0.005346,0.001636,-0.001819
2023-12-20,-0.010772,-0.007098,-0.010918
2023-12-21,-0.000770,0.007848,0.011243
2023-12-22,-0.005563,0.002780,-0.002734
2023-12-26,-0.002845,0.000214,-0.000065
...,...,...,...
2025-12-10,0.005756,-0.027738,0.016794
2025-12-11,-0.002694,0.010208,-0.006493
2025-12-12,0.000899,-0.010270,-0.017921


In [7]:
returns.simple(
    yahoo_data.closed_prices(
    ['AAPL', 'MSFT', 'AMZN'],
    '2023-12-17', 
    '2025-12-17')
           )

[*********************100%***********************]  3 of 3 completed


Ticker,AAPL,MSFT,AMZN
Date,,,
2023-12-19,0.005360,0.001637,-0.001817
2023-12-20,-0.010714,-0.007073,-0.010859
2023-12-21,-0.000770,0.007879,0.011307
2023-12-22,-0.005548,0.002784,-0.002730
2023-12-26,-0.002841,0.000214,-0.000065
...,...,...,...
2025-12-10,0.005772,-0.027357,0.016936
2025-12-11,-0.002690,0.010260,-0.006472
2025-12-12,0.000899,-0.010218,-0.017761


# Risk Engines

## Risk engine timeseries

In [49]:
class riskmetrics_timeseries:

    @staticmethod
    def var(distribution, alpha=5):
        return np.percentile(distribution, alpha)

    @staticmethod
    def cvar(distribution, alpha=5):
        var = RiskMetrics.VaR(distribution, alpha)
        return distribution[distribution <= var].mean()
    
    @staticmethod
    def var_parametric(distribution, alpha=5, dof=6):
        portofolioReturns = distribution.mean()
        portfolioStd = distribution.std()
        return np.sqrt((dof-2)/dof) * t.ppf(1-alpha/100, dof) * portfolioStd - portofolioReturns

    @staticmethod
    def cvar_parametric(distribution, alpha=5, dof=6):
        portofolioReturns = distribution.mean()
        portfolioStd = distribution.std()
        xanu = t.ppf(alpha/100, dof)
        return -1/(alpha/100) * (1-dof)**(-1) * (dof-2+xanu**2) * t.pdf(xanu, dof) * portfolioStd - portofolioReturns

In [25]:
symbols = ['AAPL', 'MSFT', 'AMZN']

In [38]:
returns_symbols = returns.logaritmic(
    yahoo_data.closed_prices(
    ['AAPL', 'MSFT', 'AMZN'],
    '2023-12-17', 
    '2025-12-17')
           )

[*********************100%***********************]  3 of 3 completed


### Usando el Risk Engine para obtener riesgo histórico de cada acción individual usando sus series de tiempo

In [39]:
for s in symbols:
    returns = returns_symbols[s]  # Extrae los retornos del símbolo específico
    var = riskmetrics_timeseries.var(returns, 5) #Extrae el var individual de cada uno
    print(f"{s} VaR (5%): {var}")

AAPL VaR (5%): -0.027420975971054917
MSFT VaR (5%): -0.022641329647592696
AMZN VaR (5%): -0.02869158155106494


In [43]:
for s in symbols:
    returns = returns_symbols[s]  # Extrae los retornos del símbolo específico
    cvar = riskmetrics_timeseries.cvar(returns, 5) #Extrae el var individual de cada uno
    print(f"{s} CVaR (5%): {cvar}")

AAPL CVaR (5%): -0.040218531527121654
MSFT CVaR (5%): -0.03227960702424511
AMZN CVaR (5%): -0.04426416089607374


### Suponiendo que las distribuciones tienen colas gordas y es necesario usar una distribución t

In [47]:
from scipy.stats import norm, t

for s in symbols:
    returns = returns_symbols[s]  # Extrae los retornos del símbolo específico
    var = riskmetrics_timeseries.var_parametric(returns, 5, 8) #Extrae el var individual de cada uno
    print(f"{s} VaR (5%): {var}")

AAPL VaR (5%): 0.027537950162892455
MSFT VaR (5%): 0.021927302004687347
AMZN VaR (5%): 0.031041081272763265


In [50]:
from scipy.stats import norm, t

for s in symbols:
    returns = returns_symbols[s]  # Extrae los retornos del símbolo específico
    cvar = riskmetrics_timeseries.cvar_parametric(returns, 5, 8) #Extrae el var individual de cada uno
    print(f"{s} CVaR (5%): {cvar}")

AAPL CVaR (5%): 0.03568358949476654
MSFT CVaR (5%): 0.028404060602901644
AMZN CVaR (5%): 0.04020933140375889


## RiskEngine Portfolio

In [58]:
class riskmetrics_portfolio:
    def var(portofolioReturns, portfolioStd, alpha=5, dof=6):
        nu = dof
        VaR = np.sqrt((nu-2)/nu) * t.ppf(1-alpha/100, nu) * portfolioStd - portofolioReturns
        return VaR
    
    def cvar(portofolioReturns, portfolioStd, alpha=5, dof=6):
        nu = dof
        xanu = t.ppf(alpha/100, nu)
        CVaR = -1/(alpha/100) * (1-nu)**(-1) * (nu-2+xanu**2) * t.pdf(xanu, nu) * portfolioStd - portofolioReturns
        return CVaR

In [59]:
riskmetrics_portfolio.var(0.15,0.07,5,3)


-0.05488994911913002

In [60]:
riskmetrics_portfolio.cvar(0.15,0.07,5,3)

0.05767364320039822

# Portfolio Optimization Engine

## Clase de pesos mínimos y máximos

In [1]:
class BoundsManager:
    '''Esta clase nos permite asignar máximos y minimos al portafolio, pero por default se manejan como min = 0 y max = 1
    Por lo que no es necesario rellenar esta parte'''
    
    def __init__(self, symbols, min_weight=0.0, max_weight=1.0):
        self.symbols = symbols
        self.min_weight = min_weight
        self.max_weight = max_weight

    def get_bounds(self):
        noa = len(self.symbols)
        return tuple((self.min_weight, self.max_weight) for _ in range(noa))

## Métricas del protafolio

In [3]:
class PortfolioMetrics:
    def __init__(self, log_returns, risk_free_rate, risk_aversion):
        self.log_returns = log_returns
        self.risk_free_rate = risk_free_rate
        self.risk_aversion = risk_aversion
        self.cov_matrix = log_returns.cov() * 252
    
    #Función para obtener el rendimiento
    def port_ret(self, weights):
        return np.sum(self.log_returns.mean() * weights) * 252
    
    #Función para obtener la volatbilidad 
    def port_vol(self, weights):
        return np.sqrt(np.dot(weights.T, np.dot(self.cov_matrix, weights)))

    #Función Objetivo: Calcula el Sharp Ratio 
    def sharpe_ratio(self, weights):
        return (self.port_ret(weights) - self.risk_free_rate) / self.port_vol(weights)

## Frontera eficiente con funciones objetivo

In [4]:
class EfficientFrontier:
    def __init__(self, symbols, metrics, bounds_manager=None):
        self.symbols = symbols
        self.metrics = metrics
        self.bounds_manager = bounds_manager or BoundsManager(symbols) 

        self.results_stats = {}
        self.results_weights = {}
    
    # = = = = = OBJETIVOS DE OPTIMIZACIÓN = = = = =
    
    # --- Optimización mínima varianza ---
    def minimize_variance(self, bounds=None):
        noa = len(self.symbols)
        bounds = self.bounds_manager.get_bounds()
        cons = ({'type':'eq','fun': lambda x: np.sum(x) - 1})
        eweights = np.array(noa * [1./noa])

        optv = sco.minimize(self.metrics.port_vol, eweights, method='SLSQP',
                            bounds=bounds, constraints=cons)

        weights = optv['x']
        ret = self.metrics.port_ret(weights)
        vol = self.metrics.port_vol(weights)
        var = vol**2
        sharpe = (ret - self.metrics.risk_free_rate) / vol
        utility = ret - (self.metrics.risk_aversion/2)*var

        # Guardar resultados
        self.results_stats['min_var'] = {
            'return': ret, 'volatility': vol, 'variance': var,
            'sharpe': sharpe, 'Exp Utility': utility
        }
        self.results_weights['min_var'] = dict(zip(self.symbols, weights.round(4)))

        return self.results_stats['min_var'], self.results_weights['min_var']

    # --- Optimización máximo Sharpe ---
    def maximize_sharpe(self, bounds=None):
        noa = len(self.symbols)
        bounds = self.bounds_manager.get_bounds()
        cons = ({'type':'eq','fun': lambda x: np.sum(x) - 1})
        eweights = np.array(noa * [1./noa])

        def min_func_sharpe(weights):
            return -(self.metrics.port_ret(weights) - self.metrics.risk_free_rate) / self.metrics.port_vol(weights)

        opts = sco.minimize(min_func_sharpe, eweights, method='SLSQP',
                            bounds=bounds, constraints=cons)

        weights = opts['x']
        ret = self.metrics.port_ret(weights)
        vol = self.metrics.port_vol(weights)
        var = vol**2
        sharpe = (ret - self.metrics.risk_free_rate) / vol
        utility = ret - (self.metrics.risk_aversion/2)*var

        # Guardar resultados
        self.results_stats['max_sharpe'] = {
            'return': ret, 'volatility': vol, 'variance': var,
            'sharpe': sharpe, 'Exp Utility': utility
        }
        self.results_weights['max_sharpe'] = dict(zip(self.symbols, weights.round(4)))

        return self.results_stats['max_sharpe'], self.results_weights['max_sharpe']

    # --- Resumen ---
    def summary(self):
        # DataFrame de estadísticas
        stats_df = pd.DataFrame(self.results_stats).T.round(4)

        # DataFrame de pesos
        weights_df = pd.DataFrame(self.results_weights).T

        return stats_df, weights_df


## Portafolio samples para las gráficas

In [5]:
import numpy as np

class PortfolioSampler:
    """
    Generates random portfolios to explore the feasible space.
    """

    def __init__(self, metrics, symbols):
        self.metrics = metrics
        self.symbols = symbols
        self.noa = len(symbols)

    def sample(self, n_portfolios: int = 3000):
        returns = np.zeros(n_portfolios)
        volatilities = np.zeros(n_portfolios)
        sharpes = np.zeros(n_portfolios)

        for i in range(n_portfolios):
            weights = np.random.random(self.noa)
            weights /= np.sum(weights)

            ret = self.metrics.port_ret(weights)
            vol = self.metrics.port_vol(weights)

            returns[i] = ret
            volatilities[i] = vol
            sharpes[i] = ret / vol

        return returns, volatilities, sharpes


# assets_info Engine

## Dataframe de estadísticas para el usuario

In [1]:
import numpy as np
import pandas as pd

def asset_statistics(returns: pd.DataFrame, risk_free_rate: float, periods: int = 252):
    
    stats = pd.DataFrame(index=returns.columns)

    # Retornos y riesgo anualizados
    mean_annual = returns.mean() * periods
    vol_annual = returns.std() * np.sqrt(periods)
    var_annual = returns.var() * periods

    stats["Return"] = mean_annual
    stats["Volatility"] = vol_annual
    stats["Variance"] = var_annual

    # Sharpe Ratio (todo en anual)
    stats["Sharpe Ratio"] = (mean_annual - risk_free_rate) / vol_annual

    # Momentos (no se anualizan)
    stats["Skewness"] = returns.skew()
    stats["Kurtosis"] = returns.kurt()

    return stats


# Montecarlo Engine

In [ ]:
import numpy as np

class PortfolioMonteCarlo:
    def __init__(
        self,
        mean_returns: np.ndarray,
        cov_matrix: np.ndarray,
        weights: np.ndarray,
        initial_value: float,
        time_horizon: int,
        n_simulations: int,
        seed: int | None = None
    ):
        self.mean_returns = mean_returns
        self.cov_matrix = cov_matrix
        self.weights = weights
        self.initial_value = initial_value
        self.time_horizon = time_horizon
        self.n_simulations = n_simulations

        if seed is not None:
            np.random.seed(seed)

        self._validate_inputs()

    def _validate_inputs(self):
        if len(self.mean_returns) != len(self.weights):
            raise ValueError("Mean returns and weights must have same length")

        if self.cov_matrix.shape[0] != self.cov_matrix.shape[1]:
            raise ValueError("Covariance matrix must be square")

        if self.cov_matrix.shape[0] != len(self.weights):
            raise ValueError("Covariance matrix size must match number of assets")

    def simulate(self) -> np.ndarray:

        # Cholesky decomposition
        chol = np.linalg.cholesky(self.cov_matrix)

        # Expected return matrix
        mean_matrix = np.tile(
            self.mean_returns, (self.time_horizon, 1)
        )

        portfolio_paths = np.zeros((self.time_horizon, self.n_simulations))

        for i in range(self.n_simulations):
            z = np.random.normal(size=(self.time_horizon, len(self.weights)))
            correlated_returns = mean_matrix + z @ chol.T
            portfolio_returns = correlated_returns @ self.weights
            portfolio_paths[:, i] = (
                np.cumprod(1 + portfolio_returns) * self.initial_value
            )

        return portfolio_paths

# Riesgo sistemático Engine

In [9]:
assets = returns.simple(
    yahoo_data.closed_prices(
    ['AAPL', 'MSFT', 'AMZN'],
    '2023-12-17', 
    '2025-12-17')
           )
rm = returns.simple(
    yahoo_data.closed_prices(
    ['SPY'],
    '2023-12-17', 
    '2025-12-17')
           )

[*********************100%***********************]  3 of 3 completed
[*********************100%***********************]  1 of 1 completed


In [14]:
def calculate_beta(returns_asset, returns_market):
    # Alinear por fechas
    data = returns_asset.align(returns_market, join="inner")
    asset, market = data

    covariance = asset.cov(market)
    variance = market.var()

    return covariance / variance

In [16]:
class SystematicRisk:

    @staticmethod
    def capm_return(beta, market_return, risk_free_rate):
        return risk_free_rate + beta * (market_return - risk_free_rate)

    @staticmethod
    def alpha(realized_return, beta, market_return, risk_free_rate):

        capm_ret = SystematicRisk.capm_return(
            beta, market_return, risk_free_rate
        )
        return realized_return - capm_ret
